In [0]:
import requests, json, os, time
from datetime import datetime
from zoneinfo import ZoneInfo

BASE_PATH = "/Volumes/workspace/bronze/raw_files/places"
ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",  # reserva (pode estar bloqueado no workspace)
]
CIDADE = "sao_paulo"
S, W, N, E = -23.80, -46.83, -23.35, -46.36  # sul, oeste, norte, leste

agora = datetime.now(ZoneInfo("America/Sao_Paulo"))
ingestion_date = agora.strftime("%Y-%m-%d")
ts = agora.strftime("%H%M%S")

# Caracteres especiais montados por código
LB, RB, LP, RP = chr(91), chr(93), chr(40), chr(41)

bbox = f"{LP}{S},{W},{N},{E}{RP}"
filtro_bairros = f'{LB}"place"~"suburb|neighbourhood|quarter"{RB}'
filtro_estacoes = f'{LB}"railway"="station"{RB}'

query = (
    f"{LB}out:json{RB}{LB}timeout:120{RB};\n"
    f"{LP}\n"
    f"  node{filtro_bairros}{bbox};\n"
    f"  node{filtro_estacoes}{bbox};\n"
    f"{RP};\n"
    f"out body;"
)
print(query)

headers = {"User-Agent": "weather-food-recommendation/0.1 (projeto de portfolio)"}

payload = None
for tentativa in range(1, 4):
    for url in ENDPOINTS:
        try:
            r = requests.post(url, data={"data": query}, headers=headers, timeout=180)
        except requests.exceptions.RequestException as ex:
            print(f"Tentativa {tentativa} | {url} -> erro de conexão ({type(ex).__name__})")
            continue

        print(f"Tentativa {tentativa} | {url} -> status {r.status_code}")

        if r.status_code == 200:
            payload = r.json()
            break

        if r.status_code == 400:
            print(r.text[:1500])
            raise ValueError("Query inválida (400). Veja a mensagem do Overpass acima.")

    if payload is not None:
        break
    time.sleep(30 * tentativa)  # 429, 502, 504: servidor ocupado

if payload is None or not payload.get("elements"):
    raise RuntimeError("Overpass não devolveu dados. Espere alguns minutos e rode a célula de novo.")

print("Elementos retornados:", len(payload["elements"]))

In [0]:
if payload is None or not payload.get("elements"):
    raise ValueError("payload vazio: rode a célula 1 e confirme status 200")

pasta = f"{BASE_PATH}/ingestion_date={ingestion_date}"
os.makedirs(pasta, exist_ok=True)

# Remove o arquivo vazio (null) gravado na tentativa anterior, se ainda existir
try:
    dbutils.fs.rm(f"{BASE_PATH}/ingestion_date=2026-10-30/sao_paulo_161249.json")
except Exception:
    pass

caminho = f"{pasta}/{CIDADE}_{ts}.json"
with open(caminho, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False)

print("Salvo em:", caminho, "| elementos:", len(payload["elements"]))
display(dbutils.fs.ls(pasta))

In [0]:
from collections import Counter

def tipo(el):
    t = el.get("tags", {})
    return t.get("place") or ("station" if t.get("railway") == "station" else "outro")

print(Counter(tipo(el) for el in payload["elements"]))

print("\nSacomã no OpenStreetMap:")
for el in payload["elements"]:
    nome = el.get("tags", {}).get("name", "")
    if "sacom" in nome.lower():
        print(" ", tipo(el), "|", nome, "|", el["lat"], el["lon"])